# MMSep Module 2 — LLaVA v1.5 7B Baseline smoke test

This notebook performs one **non-formal** image inference on free Google Colab. Run cells in order. It never uses DeepSeek, never contains an API key, and never pushes to GitHub. Model files stay under `/content`; only the final JSONL result should be saved to Drive.

Before starting, select **Runtime → Change runtime type → GPU**. Stop if the first cell reports less than 12 GB total GPU memory.

In [ ]:
# 1. Hardware gate: do not download 17+ GB of models before this passes.
import shutil, subprocess

assert shutil.which('nvidia-smi'), 'No NVIDIA GPU is attached. Change the runtime type to GPU.'
query = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader,nounits'],
    check=True, capture_output=True, text=True,
).stdout.strip().splitlines()[0]
gpu_name, total_text, free_text = [part.strip() for part in query.split(',')]
total_mib, free_mib = int(total_text), int(free_text)
print({'gpu': gpu_name, 'total_mib': total_mib, 'free_mib': free_mib})
assert total_mib >= 12 * 1024, 'This runtime has less than 12 GB VRAM; reconnect later for another GPU.'
assert free_mib >= 10 * 1024, 'Too much VRAM is already occupied. Restart the runtime before continuing.'

In [ ]:
# 2. Upload the locally reviewed mmsep_colab_source.zip bundle.
from google.colab import files
from pathlib import Path
import io, os, zipfile

PROJECT_ROOT = Path('/content/mmsep_project')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith('.zip')]
assert len(archives) == 1, 'Upload exactly one mmsep_colab_source.zip file.'
with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
    root = PROJECT_ROOT.resolve()
    for member in archive.infolist():
        target = (PROJECT_ROOT / member.filename).resolve()
        assert os.path.commonpath([root, target]) == str(root), f'Unsafe ZIP member: {member.filename}'
    archive.extractall(PROJECT_ROOT)
assert (PROJECT_ROOT / 'src/mmsep_testkit').is_dir(), 'The uploaded bundle has the wrong structure.'
print('Project source extracted to', PROJECT_ROOT)

In [ ]:
# 3. Create an isolated Python 3.10 environment; do not replace Colab's own kernel.
import os, subprocess, sys
from pathlib import Path

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv', 'modelscope-hub'], check=True)
UV_ENV = os.environ.copy()
UV_ENV.pop('UV_SYSTEM_PYTHON', None)
VENV = Path('/content/venvs/llava310')
VENV_PY = VENV / 'bin/python'
subprocess.run(['uv', 'venv', '--clear', '--python', '3.10', str(VENV)], check=True, env=UV_ENV)
source_requirements = PROJECT_ROOT / 'requirements/llava-cloud-linux.txt'
runtime_requirements = Path('/content/llava-cloud-runtime.txt')
filtered_lines = [
    line for line in source_requirements.read_text(encoding='utf-8').splitlines()
    if not line.startswith(('--extra-index-url', 'torch==', 'torchvision==', 'bitsandbytes=='))
]
runtime_requirements.write_text('\n'.join(filtered_lines) + '\n', encoding='utf-8')
subprocess.run([
    'uv', 'pip', 'install', '--python', str(VENV_PY),
    '--default-index', 'https://download.pytorch.org/whl/cu121',
    'torch==2.4.1+cu121', 'torchvision==0.19.1+cu121',
], check=True, env=UV_ENV)
subprocess.run([
    'uv', 'pip', 'install', '--python', str(VENV_PY),
    '--default-index', 'https://pypi.org/simple', 'bitsandbytes==0.48.2', '-r', str(runtime_requirements),
], check=True, env=UV_ENV)
subprocess.run(['uv', 'pip', 'check', '--python', str(VENV_PY)], check=True, env=UV_ENV)
site_packages = VENV / 'lib/python3.10/site-packages'
cuda_library_dirs = [str(path) for path in (site_packages / 'nvidia').glob('*/lib') if path.is_dir()]
if Path('/usr/local/cuda/lib64').is_dir():
    cuda_library_dirs.append('/usr/local/cuda/lib64')
existing_ld_path = os.environ.get('LD_LIBRARY_PATH', '')
if existing_ld_path:
    cuda_library_dirs.append(existing_ld_path)
os.environ['LD_LIBRARY_PATH'] = ':'.join(cuda_library_dirs)
UV_ENV['LD_LIBRARY_PATH'] = os.environ['LD_LIBRARY_PATH']
LLAVA_SOURCE = Path('/content/LLaVA-v1.2.0')
if not LLAVA_SOURCE.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', 'v1.2.0',
        'https://github.com/haotian-liu/LLaVA.git', str(LLAVA_SOURCE),
    ], check=True)
subprocess.run([
    'uv', 'pip', 'install', '--python', str(VENV_PY),
    '--no-deps', '-e', str(LLAVA_SOURCE),
], check=True, env=UV_ENV)
subprocess.run([
    str(VENV_PY), '-c',
    "import importlib.metadata as m, torch, transformers, llava; print('torch/cuda/transformers/bnb=', torch.__version__, torch.version.cuda, transformers.__version__, m.version('bitsandbytes')); print('cuda=', torch.cuda.is_available(), torch.cuda.get_device_name(0))",
], check=True, env=UV_ENV)

In [ ]:
# 4. Download the language model and the separate CLIP vision tower to ephemeral storage.
import subprocess
from pathlib import Path

MODEL_ROOT = Path('/content/models')
MODEL_PATH = MODEL_ROOT / 'llava-v1.5-7b'
VISION_PATH = MODEL_ROOT / 'clip-vit-large-patch14-336'
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
if not (MODEL_PATH / 'pytorch_model.bin.index.json').is_file():
    subprocess.run([
        'ms-hub', 'download', 'huangjianuo/llava-v1.5-7b',
        '--revision', 'master', '--local-dir', str(MODEL_PATH),
    ], check=True)
if not (VISION_PATH / 'config.json').is_file():
    subprocess.run([
        'ms-hub', 'download', 'openai-mirror/clip-vit-large-patch14-336',
        '--revision', 'master', '--local-dir', str(VISION_PATH),
    ], check=True)
print('Model:', MODEL_PATH)
print('Vision tower:', VISION_PATH)

In [ ]:
# 5. Mandatory preflight. This must finish with ok=true before inference.
import os, subprocess

BASE_CONFIG = PROJECT_ROOT / 'configs/experiments/baseline.llava.example.json'
RUN_ENV = os.environ.copy()
RUN_ENV.update({
    'PYTHONPATH': str(PROJECT_ROOT / 'src'),
    'LLAVA_MODEL_PATH': str(MODEL_PATH),
    'LLAVA_VISION_TOWER_PATH': str(VISION_PATH),
    'LLAVA_CODE_PATH': str(LLAVA_SOURCE),
    'HF_HUB_OFFLINE': '1',
    'TRANSFORMERS_OFFLINE': '1',
    'TOKENIZERS_PARALLELISM': 'false',
})
subprocess.run([
    str(VENV_PY), '-m', 'unittest', 'discover',
    '-s', str(PROJECT_ROOT / 'tests/unit'), '-v',
], env=RUN_ENV, check=True)
preflight = subprocess.run([
    str(VENV_PY), '-m', 'mmsep_testkit.preflight',
    '--config', str(BASE_CONFIG),
], env=RUN_ENV, text=True)
assert preflight.returncode == 0, 'Preflight failed. Do not run the next cell until every failed check is fixed.'

In [ ]:
# 6. Upload one ordinary test image and create a non-formal 32-token smoke case.
from google.colab import files
import json
from pathlib import Path

image_upload = files.upload()
image_names = [name for name in image_upload if Path(name).suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}]
assert len(image_names) == 1, 'Upload exactly one PNG/JPG/JPEG/WebP image.'
INPUT_DIR = Path('/content/mmsep_inputs')
INPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_PATH = INPUT_DIR / Path(image_names[0]).name
IMAGE_PATH.write_bytes(image_upload[image_names[0]])
CASE_PATH = INPUT_DIR / 'baseline_smoke.jsonl'
case = {
    'case_id': 'M2-FUN-SMOKE-001',
    'title': 'Single-image Baseline smoke test',
    'dimension': 'functional',
    'input': {'text': 'Describe the main objects and scene in this image briefly.', 'image_ref': str(IMAGE_PATH)},
    'formal': False,
    'metadata': {'source': 'manual-colab-smoke'},
}
CASE_PATH.write_text(json.dumps(case, ensure_ascii=False) + '\n', encoding='utf-8')
smoke_config = json.loads(BASE_CONFIG.read_text(encoding='utf-8'))
smoke_config['generation']['max_new_tokens'] = 32
smoke_config['experiment']['seeds'] = [42]
smoke_config['experiment']['repetitions'] = 1
SMOKE_CONFIG = INPUT_DIR / 'baseline.smoke.local.json'
SMOKE_CONFIG.write_text(json.dumps(smoke_config, ensure_ascii=False, indent=2), encoding='utf-8')
print('Image:', IMAGE_PATH)
print('Case:', CASE_PATH)

In [ ]:
# 7. Explicit execution gate: this cell loads the 7B model and runs one inference.
import json, subprocess
from pathlib import Path

RESULT_DIR = Path('/content/mmsep_results')
RESULT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_PATH = RESULT_DIR / 'baseline-smoke.jsonl'
run = subprocess.run([
    str(VENV_PY), '-m', 'mmsep_testkit.runner',
    '--config', str(SMOKE_CONFIG),
    '--cases', str(CASE_PATH),
    '--output', str(RESULT_PATH),
], env=RUN_ENV, text=True)
assert run.returncode == 0, 'Baseline inference failed; preserve the complete cell output for diagnosis.'
result = json.loads(RESULT_PATH.read_text(encoding='utf-8').splitlines()[0])
assert result.get('error') is None, f"Baseline inference returned an error: {result.get('error')}"
assert result.get('metrics', {}).get('output_tokens', 0) > 0, 'Baseline generated zero tokens.'
assert result.get('output_text', '').strip(), 'Baseline generated empty text.'
print(json.dumps({
    'case_id': result['case_id'],
    'output_text': result['output_text'],
    'duration_ms': result['duration_ms'],
    'metrics': result['metrics'],
}, ensure_ascii=False, indent=2))

In [ ]:
# 8. Optional but recommended: persist only the small result file to Google Drive.
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
drive_result_dir = Path('/content/drive/MyDrive/MMSep_test_results')
drive_result_dir.mkdir(parents=True, exist_ok=True)
saved = shutil.copy2(RESULT_PATH, drive_result_dir / RESULT_PATH.name)
print('Saved result:', saved)